In [ ]:
import random
import time

import matplotlib.pyplot as plt
from shapely import LineString, MultiPolygon, buffer
from shapely.affinity import translate
from shapely.wkt import loads as loads_wkt
from tspn_bnb2 import AnnotatedInstance
from tspn_bnb2.misc import init_params
from tspn_bnb2.operations import solve_annotated_instance
from tspn_bnb2.order_annotation import add_order_annotations

init_params()

In [ ]:
t_line = LineString([(0, 8), (5, 8), (2.5, 8), (2.5, 0)])
s_line = LineString([(5, 8), (2, 8), (0, 6), (2, 4), (3, 4), (5, 2), (3, 0), (0, 0)])
p_line = LineString([(0, 0), (0, 8), (5, 8), (5, 4), (0, 4)])
n_line = LineString([(0, 0), (0, 8), (5, 0), (5, 8)])
tspn_lines = [
    t_line,
    translate(s_line, 7, 0),
    translate(p_line, 14, 0),
    translate(n_line, 21, 0),
]
tspn_polys = [buffer(line, 0.5, quad_segs=2) for line in tspn_lines]


def randpointonside():
    side = random.randint(0, 3)
    if side == 0:
        point = random.randint(-1, 9)
        return (-1, point)
    if side == 1:
        point = random.randint(-1, 6)
        return (-1, point)
    if side == 2:
        point = random.randint(-1, 9)
        return (6, point)
    if side == 3:
        point = random.randint(-1, 6)
        return (9, point)

    raise NotImplementedError


cut_polys = []
for i, rawpoly in enumerate(tspn_polys):
    work = MultiPolygon([rawpoly])
    while len(work.geoms) < 12:
        cut_seg = translate(LineString([randpointonside(), randpointonside()]), 7 * i, 0)
        while cut_seg.length < 5:
            cut_seg = translate(LineString([randpointonside(), randpointonside()]), 7 * i, 0)
        work = work.difference(cut_seg.buffer(0.1, 1))
        if type(work) is not MultiPolygon:
            work = MultiPolygon([work])
    cut_polys.extend(list(work.geoms))


instance = AnnotatedInstance(polygons=[loads_wkt(p.wkt) for p in cut_polys])
instance = add_order_annotations(instance)


start = time.time()
sol = solve_annotated_instance(instance=instance)
end = time.time()

In [ ]:
from shapely.plotting import plot_line, plot_polygon

cmap = plt.get_cmap("tab20")
fig, ax = plt.subplots(figsize=(4, 2))
ax.set_aspect("equal")
ax.axis("off")
for j, poly in enumerate(instance.polygons):
    color = cmap(j % cmap.N)
    face_rgba = (*color[:3], 0.5)
    plot_polygon(
        poly, ax=ax, facecolor=face_rgba, edgecolor="black", add_points=False, linewidth=0.4
    )

plot_line(sol.trajectory, ax=ax, color="black", add_points=False, linewidth=1)
xs, ys = zip(*sol.trajectory.coords)
ax.scatter(xs, ys, color="black", s=5, zorder=5)

ax.set_title(
    f"{len(instance.sites())} polygons, solved in {end - start:0.2f}s",
    y=-0.3,
    fontsize=8,
)
fig.savefig("out/tspn.pdf", bbox_inches="tight")